In [ ]:
!pip install youtube-transcript-api langchain_huggingface
!pip install langchain-experimental langchain-openai chromadb langchain-chroma

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('gemini_api')

In [ ]:
# pip install youtube-transcript-api

import re
from youtube_transcript_api import YouTubeTranscriptApi


def get_video_id(url):
    """Extract YouTube video ID from URL."""

    patterns = [
        r"(?:v=)([\w-]{11})",
        r"(?:youtu\.be/)([\w-]{11})",
        r"(?:shorts/)([\w-]{11})"
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    # If video ID itself is provided
    if re.fullmatch(r"[\w-]{11}", url):
        return url

    raise ValueError("Invalid YouTube URL")


def get_youtube_transcript(url):
    """
    Extract YouTube transcript and return it as a single string.

    Preferred languages:
    English -> Hindi -> Kannada

    Returns:
        str: Timestamped transcript
    """

    video_id = get_video_id(url)

    api = YouTubeTranscriptApi()

    # Get available transcripts
    transcripts = list(api.list(video_id))

    if not transcripts:
        raise Exception("No transcript available for this video.")

    # Preferred languages
    preferred_languages = ["en", "hi", "kn"]

    selected = None

    # Find preferred language
    for language in preferred_languages:
        for transcript in transcripts:
            if transcript.language_code == language:
                selected = transcript
                break

        if selected:
            break

    # If English/Hindi/Kannada aren't available,
    # use the first available transcript
    if selected is None:
        selected = transcripts[0]

    # Fetch transcript
    transcript = selected.fetch()

    # Build a single string
    result = []

    for item in transcript:

        start = int(item.start)

        minutes = start // 60
        seconds = start % 60

        timestamp = f"[{minutes:02d}:{seconds:02d}]"

        result.append(
            f"{timestamp} {item.text}"
        )

    # Join everything into one string
    return "\n".join(result)

In [ ]:
url = input("Enter the Youtube URL:")
text = get_youtube_transcript(url)

Enter the Youtube URL:https://youtu.be/_uc0NTb7q_8?si=D3NjEAA4XGfMtuCN


In [ ]:
from langchain_core.documents import Document

docs = [Document(page_content=text)]

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

/tmp/ipykernel_15395/3094241410.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
text_splitter = SemanticChunker(embeddings=embeddings)
chunks = text_splitter.split_documents(docs)

In [ ]:
chunks

[Document(metadata={}, page_content="[00:00] ಸೋ ಗಾಯ್ಸ್ ಈಗ ಜಸ್ಟ್ ನಾವು ಮೆಟ್ರೋ ಸ್ಟೇಷನ್\n[00:02] ಗೆ ಮೆಟ್ರೋ ಸ್ಟೇಷನ್ ಇಂದ ಆಟೋದಲ್ಲಿ ನಾನು ರೇವಣ\n[00:05] ಭುವನ ಹಾಯ್\n[00:07] ಹೇಳ್ರಿ ಮೆಟ್ರೋ\n[00:10] ಸ್ಟೇಷನ್ ಟ್ರೈ ಮಾಡೋಣ ಅಲ್ಲೇ ಸಿಗೋಣ ಸೋ ಈಗ\n[00:15] ಸ್ವಲ್ಪ ಟ್ರಾಫಿಕ್ ನ ತಿಂತಿದ್ದೀವಿ\n[00:19] ಸ್ವಲ್ಪ ಲೇಟ್ ಆಯ್ತು ಅಷ್ಟೇ ಈಗ ಟೈಮ್\n[00:21] ಎಷ್ಟಾಗಿದೆ 11 ಸಂಥಿಂಗ್ ಆಗಿದೆ ಎಲ್ಲಾ ಅಲ್ಲ\n[00:24] 10:30 ಆಗಿದೆ ಸಾರಿ ಈಗ ನಾವು ಹೋಗೋಕೆ ಒಂದು\n[00:26] 11:00 ಆಗ್ತದೆ ಆಮೇಲೆ ಅಲ್ಲಿ ಎಲ್ಲಾ\n[00:28] ತೋರಿಸ್ತೀನಿ ಟ್ರಾಫಿಕ್ ನಲ್ಲಿ ಏನು ಮೇಲೆ ಈ\n[00:30] ಟ್ರಾಫಿಕ್ ಕ್ಲಿಯರ್ ಆಗೋದು ಆಟೋದಾಗ\n[00:32] ಒಂಟಿ ಈಗ ಯಶವಂತಪುರದಲ್ಲಿ ಹಾಕ್ತೀನಿ ಇನ್ನೊಂದು\n[00:35] ಅದೇ ಇಂಗ್ಲಿಷ್ ಪ್ಯಾಲೆಸ್ ಗೌರ್ನಿಂಗ್ ಫುಲ್\n[00:37] ಬ್ಲಾಗ್ ಮಾಡ್ತೀನಿ ಇಂಗ್ಲಿಷ್ ನ ಮಾಡ್ತೀನಿ ಎಚ್\n[00:39] ಡಿ ಗಿಚ್ಚು ತೋರಿಸಿರ್ತೀನಿ ಓಕೆ ಅಡ್ಜಸ್ಟ್\n[00:42] ಮಾಡಿರಿ ಇಂಗ್ಲಿಷ್ ಅರ್ಥ ಇಲ್ಲ ಅಂದ್ರೆ ಹಂಗೆ\n[00:44] ಅರ್ಥ ಮಾಡಿಕೊಳ್ಳಿರಿ ನನ್ನ ಪಕ್ಕದಲ್ಲಿ\n[00:49] ಇಂಗ್ಲಿಷ್ ಗಾಯ್ಸ್ ಈಗ ಜಸ್ಟ್ ಒಳಗೆ ಎಂಟರ್\n[00:52] ಆಗಿದ್ದೀವಿ ಎಲ್ಲಿ ಅಂದ್ರೆ ಪ್ಯಾಲೆಸ್ ತಗೊಂಡು\n[00:54] ಪ್ಯಾಲೆಸ್\n[00:59] [ಸಂಗೀತ]\n[01:59] اللہ\n[02:12] ಸೋ ಗಾಯ್ಸ್ ಈಗ ಒಳಗೆ ಎಂಟರ್ ಆಗಿದ್ದೀವಿ ಈಗ\n[02:14] ಮೈಸೂರ

In [ ]:
import os
import shutil
if os.path.exists("vector.db"):
    shutil.rmtree("vector.db")
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="vector.db")
else:
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="vector.db")

In [ ]:
retriver = vectorstore.as_retriever(
    search_typr="mmr",
    search_kwargs={"k": 4}
)

In [ ]:
retriver.invoke("let her go")

[Document(id='02263cdd-2068-46dd-b508-1c1cb24bb666', metadata={}, page_content="[00:00] ಸೋ ಗಾಯ್ಸ್ ಈಗ ಜಸ್ಟ್ ನಾವು ಮೆಟ್ರೋ ಸ್ಟೇಷನ್\n[00:02] ಗೆ ಮೆಟ್ರೋ ಸ್ಟೇಷನ್ ಇಂದ ಆಟೋದಲ್ಲಿ ನಾನು ರೇವಣ\n[00:05] ಭುವನ ಹಾಯ್\n[00:07] ಹೇಳ್ರಿ ಮೆಟ್ರೋ\n[00:10] ಸ್ಟೇಷನ್ ಟ್ರೈ ಮಾಡೋಣ ಅಲ್ಲೇ ಸಿಗೋಣ ಸೋ ಈಗ\n[00:15] ಸ್ವಲ್ಪ ಟ್ರಾಫಿಕ್ ನ ತಿಂತಿದ್ದೀವಿ\n[00:19] ಸ್ವಲ್ಪ ಲೇಟ್ ಆಯ್ತು ಅಷ್ಟೇ ಈಗ ಟೈಮ್\n[00:21] ಎಷ್ಟಾಗಿದೆ 11 ಸಂಥಿಂಗ್ ಆಗಿದೆ ಎಲ್ಲಾ ಅಲ್ಲ\n[00:24] 10:30 ಆಗಿದೆ ಸಾರಿ ಈಗ ನಾವು ಹೋಗೋಕೆ ಒಂದು\n[00:26] 11:00 ಆಗ್ತದೆ ಆಮೇಲೆ ಅಲ್ಲಿ ಎಲ್ಲಾ\n[00:28] ತೋರಿಸ್ತೀನಿ ಟ್ರಾಫಿಕ್ ನಲ್ಲಿ ಏನು ಮೇಲೆ ಈ\n[00:30] ಟ್ರಾಫಿಕ್ ಕ್ಲಿಯರ್ ಆಗೋದು ಆಟೋದಾಗ\n[00:32] ಒಂಟಿ ಈಗ ಯಶವಂತಪುರದಲ್ಲಿ ಹಾಕ್ತೀನಿ ಇನ್ನೊಂದು\n[00:35] ಅದೇ ಇಂಗ್ಲಿಷ್ ಪ್ಯಾಲೆಸ್ ಗೌರ್ನಿಂಗ್ ಫುಲ್\n[00:37] ಬ್ಲಾಗ್ ಮಾಡ್ತೀನಿ ಇಂಗ್ಲಿಷ್ ನ ಮಾಡ್ತೀನಿ ಎಚ್\n[00:39] ಡಿ ಗಿಚ್ಚು ತೋರಿಸಿರ್ತೀನಿ ಓಕೆ ಅಡ್ಜಸ್ಟ್\n[00:42] ಮಾಡಿರಿ ಇಂಗ್ಲಿಷ್ ಅರ್ಥ ಇಲ್ಲ ಅಂದ್ರೆ ಹಂಗೆ\n[00:44] ಅರ್ಥ ಮಾಡಿಕೊಳ್ಳಿರಿ ನನ್ನ ಪಕ್ಕದಲ್ಲಿ\n[00:49] ಇಂಗ್ಲಿಷ್ ಗಾಯ್ಸ್ ಈಗ ಜಸ್ಟ್ ಒಳಗೆ ಎಂಟರ್\n[00:52] ಆಗಿದ್ದೀವಿ ಎಲ್ಲಿ ಅಂದ್ರೆ ಪ್ಯಾಲೆಸ್ ತಗೊಂಡು\n[00:54] ಪ್ಯಾಲೆಸ್\n[00:59] [ಸಂಗೀತ]\n[01:59] اللہ\n[02:12] ಸೋ ಗಾಯ್ಸ

In [ ]:
from openai import OpenAI
model = OpenAI(base_url="https://openrouter.ai/api/v1",api_key="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")

In [ ]:
def answer_question(question: str, history):
    # 1. Standardise history structure
    history_messages = [{"role": h["role"], "content": h["content"]} for h in history]

    # 2. Retrieve Documents directly using the raw question
    docs = retriver.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # 3. Answer the question using the retrieved context via OpenRouter
    SYSTEM_PROMPT_TEMPLATE = f"""
You are a knowledgeable and friendly AI assistant. Answer the user's questions using only the relevant context or history from the provided PDF.

Context:
{context}

If the answer is not found in or related to the context, politely say so. Do not make up information.
"""

    final_messages = (
        [{"role": "system", "content": SYSTEM_PROMPT_TEMPLATE}]
        + history_messages
        + [{"role": "user", "content": question}]
    )

    return model.chat.completions.create(
        model="openrouter/free",  # Explicitly using a specific free model string
        messages=final_messages
    ).choices[0].message.content


In [ ]:
import gradio as gr

In [ ]:
gr.ChatInterface(answer_question).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://afdedbc238ef4ae04e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
answer_question("hi",[])

"Hello! I'm not able to find information about greetings in the given video transcript, but I'm happy to help with anything else you might need, such as explaining the video's content about AI data science, programming tasks, or anything else. What would you like to talk about?"

In [ ]:
retriver.invoke("hi")

[Document(id='02263cdd-2068-46dd-b508-1c1cb24bb666', metadata={}, page_content="[00:00] ಸೋ ಗಾಯ್ಸ್ ಈಗ ಜಸ್ಟ್ ನಾವು ಮೆಟ್ರೋ ಸ್ಟೇಷನ್\n[00:02] ಗೆ ಮೆಟ್ರೋ ಸ್ಟೇಷನ್ ಇಂದ ಆಟೋದಲ್ಲಿ ನಾನು ರೇವಣ\n[00:05] ಭುವನ ಹಾಯ್\n[00:07] ಹೇಳ್ರಿ ಮೆಟ್ರೋ\n[00:10] ಸ್ಟೇಷನ್ ಟ್ರೈ ಮಾಡೋಣ ಅಲ್ಲೇ ಸಿಗೋಣ ಸೋ ಈಗ\n[00:15] ಸ್ವಲ್ಪ ಟ್ರಾಫಿಕ್ ನ ತಿಂತಿದ್ದೀವಿ\n[00:19] ಸ್ವಲ್ಪ ಲೇಟ್ ಆಯ್ತು ಅಷ್ಟೇ ಈಗ ಟೈಮ್\n[00:21] ಎಷ್ಟಾಗಿದೆ 11 ಸಂಥಿಂಗ್ ಆಗಿದೆ ಎಲ್ಲಾ ಅಲ್ಲ\n[00:24] 10:30 ಆಗಿದೆ ಸಾರಿ ಈಗ ನಾವು ಹೋಗೋಕೆ ಒಂದು\n[00:26] 11:00 ಆಗ್ತದೆ ಆಮೇಲೆ ಅಲ್ಲಿ ಎಲ್ಲಾ\n[00:28] ತೋರಿಸ್ತೀನಿ ಟ್ರಾಫಿಕ್ ನಲ್ಲಿ ಏನು ಮೇಲೆ ಈ\n[00:30] ಟ್ರಾಫಿಕ್ ಕ್ಲಿಯರ್ ಆಗೋದು ಆಟೋದಾಗ\n[00:32] ಒಂಟಿ ಈಗ ಯಶವಂತಪುರದಲ್ಲಿ ಹಾಕ್ತೀನಿ ಇನ್ನೊಂದು\n[00:35] ಅದೇ ಇಂಗ್ಲಿಷ್ ಪ್ಯಾಲೆಸ್ ಗೌರ್ನಿಂಗ್ ಫುಲ್\n[00:37] ಬ್ಲಾಗ್ ಮಾಡ್ತೀನಿ ಇಂಗ್ಲಿಷ್ ನ ಮಾಡ್ತೀನಿ ಎಚ್\n[00:39] ಡಿ ಗಿಚ್ಚು ತೋರಿಸಿರ್ತೀನಿ ಓಕೆ ಅಡ್ಜಸ್ಟ್\n[00:42] ಮಾಡಿರಿ ಇಂಗ್ಲಿಷ್ ಅರ್ಥ ಇಲ್ಲ ಅಂದ್ರೆ ಹಂಗೆ\n[00:44] ಅರ್ಥ ಮಾಡಿಕೊಳ್ಳಿರಿ ನನ್ನ ಪಕ್ಕದಲ್ಲಿ\n[00:49] ಇಂಗ್ಲಿಷ್ ಗಾಯ್ಸ್ ಈಗ ಜಸ್ಟ್ ಒಳಗೆ ಎಂಟರ್\n[00:52] ಆಗಿದ್ದೀವಿ ಎಲ್ಲಿ ಅಂದ್ರೆ ಪ್ಯಾಲೆಸ್ ತಗೊಂಡು\n[00:54] ಪ್ಯಾಲೆಸ್\n[00:59] [ಸಂಗೀತ]\n[01:59] اللہ\n[02:12] ಸೋ ಗಾಯ್ಸ